# Build API Model Artifacts

This notebook trains and saves the model artifacts needed by the service, one folder per coin ticker, for example `artifacts/models/BTC-USD/`.

Artifacts produced per coin:
- `quantile_model_bundle.joblib`
- `regime_ae.pt`
- `regime_scaler.pkl`
- `regime_train_config.pkl`

The saved path matches the current API expectation: `artifacts/models/`.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import os
import json
import joblib
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if not (ROOT / 'core').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from core.storage.coin_repository import get_coin_repository
from core.storage.market_data_repository import get_market_data_repository
from core.models.probabilistic_quantile import (
    load_features_csv,
    add_next_day_target,
    prepare_model_frame,
    time_split,
    fit_quantile_models,
)
from core.regime_detection.historical_matching import (
    FEATURE_COLUMNS,
    load_feature_data,
    scale_features,
    build_rolling_windows,
    train_autoencoder,
    save_regime_artifacts,
)

print('Project root:', ROOT)


Project root: C:\Users\sia\Desktop\capstone\src\Agentic-Crypto-Return-Service


In [2]:
COIN_REPO = get_coin_repository()
MARKET_REPO = get_market_data_repository()

# Optional explicit subset, e.g. ['BTC', 'ETH']
COIN_SYMBOLS = None

# Save path expected by the current API
ARTIFACTS_ROOT = ROOT / 'artifacts' / 'models'
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

TRAIN_FRAC = 0.80
QUANTILES = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
RANDOM_STATE = 42
N_ESTIMATORS = 200
LEARNING_RATE = 0.05
MAX_DEPTH = 3

MATCH_WINDOW_DAYS = 30
LATENT_DIM = 16
TRAIN_EPOCHS = 40
TOP_N = 10
SIMILARITY_METRIC = 'cosine'
EMBARGO_DAYS = 5

candidate_symbols = [str(s).upper() for s in COIN_SYMBOLS] if COIN_SYMBOLS else sorted(COIN_REPO.list_symbols())
print('Artifacts root:', ARTIFACTS_ROOT)
print('Candidate symbols:', candidate_symbols)


Artifacts root: C:\Users\sia\Desktop\capstone\src\Agentic-Crypto-Return-Service\artifacts\models
Candidate symbols: ['ADA', 'AVAX', 'BNB', 'BTC', 'DOGE', 'ETH', 'FLOKI', 'LINK', 'SOL', 'XRP']


In [3]:
def load_features(symbol: str) -> pd.DataFrame:
    df = load_features_csv(MARKET_REPO.read_features(symbol=symbol))
    if df.empty:
        return df
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
    return df


def train_quantile_artifact(symbol: str, out_root: Path) -> dict:
    df = load_features(symbol)
    if df.empty:
        raise FileNotFoundError(f'No features found for symbol={symbol}')

    df = add_next_day_target(df, ret_col='log_ret_1d')
    model_df, feats, target_col = prepare_model_frame(df)
    train_df, test_df = time_split(model_df, train_frac=TRAIN_FRAC)

    bundle = fit_quantile_models(
        train_df=train_df,
        feature_cols=feats,
        target_col=target_col,
        quantiles=QUANTILES,
        random_state=RANDOM_STATE,
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        max_depth=MAX_DEPTH,
    )

    ticker = str(df['ticker'].dropna().iloc[-1]) if 'ticker' in df.columns else COIN_REPO.get_by_symbol(symbol).yahoo_ticker
    coin_dir = out_root / ticker
    coin_dir.mkdir(parents=True, exist_ok=True)
    out_path = coin_dir / 'quantile_model_bundle.joblib'

    joblib.dump(
        {
            'ticker': ticker,
            'bundle': bundle,
            'feature_cols': feats,
            'target_col': target_col,
            'train_rows': len(train_df),
            'test_rows': len(test_df),
            'source_symbol': symbol.upper(),
            'hyperparameters': {
                'train_frac': TRAIN_FRAC,
                'quantiles': QUANTILES,
                'random_state': RANDOM_STATE,
                'n_estimators': N_ESTIMATORS,
                'learning_rate': LEARNING_RATE,
                'max_depth': MAX_DEPTH,
                'feature_cols': feats,
            },
        },
        out_path,
    )

    return {
        'symbol': symbol,
        'ticker': ticker,
        'artifact_type': 'quantile',
        'path': str(out_path),
        'train_rows': int(len(train_df)),
        'test_rows': int(len(test_df)),
    }


def train_regime_artifacts(symbol: str, out_root: Path) -> dict:
    df = load_features(symbol)
    if df.empty:
        raise FileNotFoundError(f'No features found for symbol={symbol}')

    ticker = str(df['ticker'].dropna().iloc[-1]) if 'ticker' in df.columns else COIN_REPO.get_by_symbol(symbol).yahoo_ticker
    df_raw = load_feature_data(df)
    df_scaled, scaler = scale_features(df_raw)
    windows, _ = build_rolling_windows(df_scaled=df_scaled, match_window_days=MATCH_WINDOW_DAYS)

    if windows.size == 0:
        raise ValueError(f'No valid regime windows were built for symbol={symbol}')

    model, _ = train_autoencoder(
        windows=windows,
        latent_dim=LATENT_DIM,
        epochs=TRAIN_EPOCHS,
        batch_size=64,
        lr=1e-3,
        seed=RANDOM_STATE,
    )

    train_cfg = {
        'match_window_days': int(MATCH_WINDOW_DAYS),
        'similarity_metric': str(SIMILARITY_METRIC),
        'embargo_days': int(EMBARGO_DAYS),
        'latent_dim': int(LATENT_DIM),
        'top_n': int(TOP_N),
        'feature_columns': list(FEATURE_COLUMNS),
    }

    save_regime_artifacts(
        ticker=ticker,
        model=model,
        scaler=scaler,
        train_config=train_cfg,
        out_dir=str(out_root),
    )

    coin_dir = out_root / ticker
    return {
        'symbol': symbol,
        'ticker': ticker,
        'artifact_type': 'regime',
        'path': str(coin_dir),
        'n_windows': int(len(windows)),
        'match_window_days': int(MATCH_WINDOW_DAYS),
        'latent_dim': int(LATENT_DIM),
    }


In [4]:
train_rows = []
train_errors = []

for symbol in candidate_symbols:
    try:
        q_row = train_quantile_artifact(symbol, ARTIFACTS_ROOT)
        train_rows.append(q_row)
        print(f"[OK] quantile artifact saved for {q_row['ticker']}: {q_row['path']}")
    except Exception as e:
        train_errors.append({'symbol': symbol, 'stage': 'quantile', 'error': str(e)})
        print(f"[ERR] quantile artifact failed for {symbol}: {e}")
        continue

    try:
        r_row = train_regime_artifacts(symbol, ARTIFACTS_ROOT)
        train_rows.append(r_row)
        print(f"[OK] regime artifacts saved for {r_row['ticker']}: {r_row['path']}")
    except Exception as e:
        train_errors.append({'symbol': symbol, 'stage': 'regime', 'error': str(e)})
        print(f"[ERR] regime artifacts failed for {symbol}: {e}")

artifact_summary_df = pd.DataFrame(train_rows)
artifact_errors_df = pd.DataFrame(train_errors)

print('\nArtifacts built:', len(artifact_summary_df))
print('Errors:', len(artifact_errors_df))
display(artifact_summary_df)
if not artifact_errors_df.empty:
    display(artifact_errors_df)


[OK] quantile artifact saved for ADA-USD: C:\Users\sia\Desktop\capstone\src\Agentic-Crypto-Return-Service\artifacts\models\ADA-USD\quantile_model_bundle.joblib
[AE] epoch 1/40 loss=0.785385
[AE] epoch 10/40 loss=0.296540
[AE] epoch 20/40 loss=0.255868
[AE] epoch 30/40 loss=0.240833
[AE] epoch 40/40 loss=0.233852
[OK] regime artifacts saved for ADA-USD: C:\Users\sia\Desktop\capstone\src\Agentic-Crypto-Return-Service\artifacts\models\ADA-USD
[OK] quantile artifact saved for AVAX-USD: C:\Users\sia\Desktop\capstone\src\Agentic-Crypto-Return-Service\artifacts\models\AVAX-USD\quantile_model_bundle.joblib
[AE] epoch 1/40 loss=0.921820
[AE] epoch 10/40 loss=0.354410
[AE] epoch 20/40 loss=0.281651
[AE] epoch 30/40 loss=0.258011
[AE] epoch 40/40 loss=0.248345
[OK] regime artifacts saved for AVAX-USD: C:\Users\sia\Desktop\capstone\src\Agentic-Crypto-Return-Service\artifacts\models\AVAX-USD
[OK] quantile artifact saved for BNB-USD: C:\Users\sia\Desktop\capstone\src\Agentic-Crypto-Return-Service\ar

,symbol,ticker,artifact_type,path,train_rows,test_rows,n_windows,match_window_days,latent_dim
0,ADA,ADA-USD,quantile,C:\Users\sia\Desktop\capstone\src\Agentic-Cryp...,2413.0,604.0,NaN,NaN,NaN
1,ADA,ADA-USD,regime,C:\Users\sia\Desktop\capstone\src\Agentic-Cryp...,NaN,NaN,2988.0,30.0,16.0
2,AVAX,AVAX-USD,quantile,C:\Users\sia\Desktop\capstone\src\Agentic-Cryp...,1576.0,395.0,NaN,NaN,NaN
3,AVAX,AVAX-USD,regime,C:\Users\sia\Desktop\capstone\src\Agentic-Cryp...,NaN,NaN,1942.0,30.0,16.0
4,BNB,BNB-USD,quantile,C:\Users\sia\Desktop\capstone\src\Agentic-Cryp...,2413.0,604.0,NaN,NaN,NaN
5,BNB,BNB-USD,regime,C:\Users\sia\Desktop\capstone\src\Agentic-Cryp...,NaN,NaN,2988.0,30.0,16.0
6,BTC,BTC-USD,quantile,C:\Users\sia\Desktop\capstone\src\Agentic-Cryp...,3332.0,834.0,NaN,NaN,NaN
7,BTC,BTC-USD,regime,C:\Users\sia\Desktop\capstone\src\Agentic-Cryp...,NaN,NaN,4137.0,30.0,16.0
8,DOGE,DOGE-USD,quantile,C:\Users\sia\Desktop\capstone\src\Agentic-Cryp...,2413.0,604.0,NaN,NaN,NaN
9,DOGE,DOGE-USD,regime,C:\Users\sia\Desktop\capstone\src\Agentic-Cryp...,NaN,NaN,2988.0,30.0,16.0


In [5]:
verification_rows = []

for ticker_dir in sorted([p for p in ARTIFACTS_ROOT.iterdir() if p.is_dir()], key=lambda p: p.name):
    verification_rows.append({
        'ticker': ticker_dir.name,
        'quantile_bundle_exists': (ticker_dir / 'quantile_model_bundle.joblib').exists(),
        'regime_ae_exists': (ticker_dir / 'regime_ae.pt').exists(),
        'regime_scaler_exists': (ticker_dir / 'regime_scaler.pkl').exists(),
        'regime_train_config_exists': (ticker_dir / 'regime_train_config.pkl').exists(),
    })

verification_df = pd.DataFrame(verification_rows).sort_values('ticker').reset_index(drop=True) if verification_rows else pd.DataFrame()
display(verification_df)


,ticker,quantile_bundle_exists,regime_ae_exists,regime_scaler_exists,regime_train_config_exists
0,ADA-USD,True,True,True,True
1,AVAX-USD,True,True,True,True
2,BNB-USD,True,True,True,True
3,BTC-USD,True,True,True,True
4,DOGE-USD,True,True,True,True
5,ETH-USD,True,True,True,True
6,FLOKI-USD,True,True,True,True
7,LINK-USD,True,True,True,True
8,SOL-USD,True,True,True,True
9,XRP-USD,True,True,True,True
